# paper-rag: Exploratory Data Analysis & Retrieval Inspection

This notebook explores the sampled **20,000 arXiv abstracts** dataset and demonstrates the comparative performance of the three retrieval strategies:
1. **BM25 Baseline** (Lexical matching)
2. **Dense Retriever** (`all-MiniLM-L6-v2` + FAISS)
3. **Hybrid Retriever** (Reciprocal Rank Fusion - RRF)


In [1]:
import re
from collections import Counter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.config import RAW_DATA_PATH, EVAL_DATA_PATH
from src.retrieval import get_retriever

print(f"Loading dataset from: {RAW_DATA_PATH}")
df = pd.read_parquet(RAW_DATA_PATH)
print(f"Dataset shape: {df.shape}")
df.head(3)

## 1. Document Length & Token Distributions

In [2]:
df['title_word_count'] = df['title'].apply(lambda x: len(x.split()))
df['abstract_word_count'] = df['abstract'].apply(lambda x: len(x.split()))

print("=== Title Word Count Statistics ===")
print(df['title_word_count'].describe())

print("\n=== Abstract Word Count Statistics ===")
print(df['abstract_word_count'].describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['title_word_count'], bins=30, color='#3b82f6', edgecolor='black')
axes[0].set_title("Distribution of Title Lengths (Words)")
axes[0].set_xlabel("Word Count")
axes[0].set_ylabel("Number of Papers")

axes[1].hist(df['abstract_word_count'], bins=40, color='#10b981', edgecolor='black')
axes[1].set_title("Distribution of Abstract Lengths (Words)")
axes[1].set_xlabel("Word Count")
axes[1].set_ylabel("Number of Papers")
plt.tight_layout()
plt.show()

## 2. Subject Category Breakdown

In [3]:
all_categories = []
for cats in df['categories'].dropna():
    all_categories.extend(cats.split())

top_categories = Counter(all_categories).most_common(15)
cat_names, cat_counts = zip(*top_categories)

plt.figure(figsize=(10, 5))
plt.barh(cat_names[::-1], cat_counts[::-1], color='#8b5cf6', edgecolor='black')
plt.xlabel("Paper Count")
plt.title("Top 15 arXiv Subject Categories in Sample")
plt.tight_layout()
plt.show()

## 3. Qualitative Retrieval Comparison: BM25 vs. Dense vs. Hybrid

In [4]:
# Initialize retrievers
bm25 = get_retriever('bm25')
dense = get_retriever('dense')
hybrid = get_retriever('hybrid', bm25_retriever=bm25, dense_retriever=dense)

test_query = "reinforcement learning for robotic control"
print(f"Retrieving for query: '{test_query}'\n")

def print_top_results(name, results):
    print(f"--- {name} (Top 2) ---")
    for r in results[:2]:
        print(f"[{r['rank']}] {r['title']} (Score: {r['score']:.4f})")
        print(f"    {r['abstract'][:140]}...\n")

print_top_results("BM25 Lexical Retriever", bm25.retrieve(test_query, top_k=2))
print_top_results("Dense FAISS Retriever", dense.retrieve(test_query, top_k=2))
print_top_results("Hybrid RRF Retriever", hybrid.retrieve(test_query, top_k=2))